# Notebook 03: DDP Deep Dive

## Why This Matters

PyTorch DistributedDataParallel (DDP) is the workhorse of multi-GPU training. It is what powers most production training runs for models up to ~10B parameters. Understanding DDP's internals -- gradient bucketing, backward-communication overlap, DistributedSampler -- helps you diagnose training hangs, debug NCCL errors, and squeeze maximum utilization from your hardware. This is the most commonly tested distributed training topic in ML engineering interviews.


In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import os, time

n_gpus = torch.cuda.device_count()
print(f"Available GPUs: {n_gpus}")
print("Ready.")

## 1. DDP Lifecycle

DDP involves these steps:
1. **`init_process_group`**: set up the distributed communication backend (NCCL for GPU, Gloo for CPU)
2. **Model wrapping**: `model = DDP(model, device_ids=[rank])` -- hooks into backward pass
3. **Forward pass**: runs identically on each rank, independent
4. **Backward pass**: DDP hooks fire, gradients are accumulated into buckets, then all-reduced
5. **Optimizer step**: `optimizer.step()` -- runs on each rank with identical, averaged gradients

The key insight: **DDP is a backward hook** -- it intercepts gradient accumulation and replaces local gradients with globally averaged ones.


In [ ]:
# DDP training function (runs inside spawn)
def ddp_worker(rank, world_size, return_dict):
    # Initialize process group
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    
    backend = 'nccl' if torch.cuda.is_available() else 'gloo'
    dist.init_process_group(backend=backend, rank=rank, world_size=world_size)
    
    device = torch.device(f'cuda:{rank}' if torch.cuda.is_available() else 'cpu')
    
    # Model and optimizer
    model = nn.Sequential(
        nn.Linear(32, 64), nn.ReLU(),
        nn.Linear(64, 10)
    ).to(device)
    
    # Wrap with DDP -- this adds backward hooks
    model = DDP(model, device_ids=[rank] if torch.cuda.is_available() else None)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    # Synthetic data
    torch.manual_seed(rank)  # different data per rank (simulating DistributedSampler)
    X = torch.randn(128, 32, device=device)
    Y = torch.randint(0, 10, (128,), device=device)
    
    # Training step
    model.train()
    losses = []
    for step in range(10):
        optimizer.zero_grad()
        logits = model(X)
        loss = nn.functional.cross_entropy(logits, Y)
        loss.backward()  # triggers DDP all-reduce here
        optimizer.step()
        losses.append(loss.item())
    
    # After DDP: all ranks have the same model weights
    w0 = model.module.state_dict()['0.weight'].clone()
    return_dict[rank] = {'losses': losses, 'weight_sample': w0[0, :4].tolist()}
    
    dist.destroy_process_group()

# Run DDP with 2 processes (works on both GPU and CPU)
if __name__ == '__main__' or True:  # notebook-safe
    world_size = min(2, max(1, n_gpus))  # use at most 2 GPUs
    manager = mp.Manager()
    return_dict = manager.dict()
    
    mp.spawn(ddp_worker, args=(world_size, return_dict), nprocs=world_size, join=True)
    
    print(f'DDP training with {world_size} process(es):')
    for rank, data in sorted(return_dict.items()):
        print(f'  Rank {rank}: final loss = {data["losses"][-1]:.4f}')
        print(f'    Weight sample: {data["weight_sample"]}')
    
    if len(return_dict) > 1:
        w0 = return_dict[0]['weight_sample']
        w1 = return_dict[1]['weight_sample']
        print(f'  Weights match across ranks: {all(abs(a-b) < 1e-5 for a,b in zip(w0, w1))}')

## 2. Gradient Bucketing: Why DDP Accumulates Before All-Reduce

Naive approach: all-reduce every gradient tensor as soon as its backward pass completes. Problem: thousands of small all-reduce calls saturate NVLink/InfiniBand with tiny messages (high latency, low bandwidth utilization).

DDP's solution: **gradient bucketing**. Gradients are accumulated into buckets of size ~25 MB (configurable). When a bucket fills, it is all-reduced as a single large message. This dramatically improves NVLink/InfiniBand utilization.

The order matters: DDP processes buckets in **reverse layer order** (last layers computed first, matching backward pass order). This enables overlap.


In [ ]:
# Gradient bucketing: visualize the concept
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Simulate a model with many small tensors vs few large buckets
np.random.seed(42)
n_layers = 20
param_sizes_mb = np.random.exponential(2.0, n_layers)  # MB per layer
param_sizes_mb = np.clip(param_sizes_mb, 0.1, 10.0)

# Without bucketing: N all-reduce calls
bucket_size_mb = 25.0
buckets = []
current_bucket = 0
bucket_assignments = []
for s in param_sizes_mb:
    bucket_assignments.append(len(buckets))
    current_bucket += s
    if current_bucket >= bucket_size_mb:
        buckets.append(current_bucket)
        current_bucket = 0
if current_bucket > 0:
    buckets.append(current_bucket)

# Plot
ax = axes[0]
ax.bar(range(n_layers), param_sizes_mb, color='steelblue', alpha=0.7)
ax.set_xlabel('Layer index')
ax.set_ylabel('Parameter size (MB)')
ax.set_title(f'Without bucketing: {n_layers} all-reduce calls')
ax.grid(True, alpha=0.3)

ax = axes[1]
colors = plt.cm.tab10(np.linspace(0, 1, len(buckets)))
for layer_i, bucket_i in enumerate(bucket_assignments):
    ax.bar(layer_i, param_sizes_mb[layer_i], color=colors[bucket_i], alpha=0.8)
ax.set_xlabel('Layer index')
ax.set_ylabel('Parameter size (MB)')
ax.set_title(f'With bucketing ({bucket_size_mb} MB): {len(buckets)} all-reduce calls')
# Legend for buckets
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], label=f'Bucket {i}: {buckets[i]:.1f} MB')
                   for i in range(len(buckets))]
ax.legend(handles=legend_elements, loc='upper right', fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gradient_bucketing.png', dpi=100)
plt.show()

print(f'Without bucketing: {n_layers} separate all-reduce calls')
print(f'With bucketing: {len(buckets)} batched all-reduce calls')
print(f'Reduction in comm calls: {n_layers / len(buckets):.1f}x')

## 3. Communication-Computation Overlap

DDP's biggest performance trick: **overlap backward computation with gradient communication**.

Timeline without overlap:
```
[forward] → [backward] → [all-reduce] → [optimizer]
```

Timeline with overlap:
```
[forward] → [backward layer N ... layer K] → [all-reduce bucket N]
                                          → [backward layer K-1 ... ]
```

As soon as the last layer in a bucket finishes its backward pass, the all-reduce for that bucket starts. Meanwhile, the rest of the backward pass continues. The network stays busy while the GPU keeps computing.

This overlap is automatic in DDP -- enabled by the bucket backward hooks.


In [ ]:
# DDP configuration options
model = nn.Sequential(
    nn.Linear(128, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10)
)

print('DDP configuration options:')
print('\n1. bucket_cap_mb (default=25): trade-off between latency and bandwidth')
print('   Smaller: more frequent, smaller all-reduces (more latency overhead)')
print('   Larger: fewer, larger all-reduces (more overlap opportunity)')

print('\n2. find_unused_parameters (default=False):')
print('   Set True if some parameters may not receive gradients every step')
print('   (e.g., conditional computation, mixture-of-experts)')
print('   Cost: extra traversal pass at each backward')
print('   Do NOT set True if not needed -- significant overhead')

print('\n3. gradient_as_bucket_view (default=False):')
print('   Avoids extra memory copy by using bucket memory as gradient storage')
print('   Saves ~params_size bytes of memory')

print('\n4. static_graph (default=False):')
print('   Enable if your model has the same computational graph every iteration')
print('   Allows DDP to skip graph traversal overhead')

# DistributedSampler: ensures each rank sees different data
print('\n5. DistributedSampler: crucial for correct training')
print('   Without it: all ranks see the same data -> no benefit from data parallelism')
print('   With it: data is partitioned; effective batch size = local_batch * world_size')

dataset = TensorDataset(torch.randn(1000, 32), torch.randint(0, 10, (1000,)))
print(f'\nExample: dataset of 1000 samples, 4 ranks')
print(f'  Each rank sees 1000/4 = 250 samples per epoch')
print(f'  Total gradient: average over all 1000 samples (via all-reduce)')

## 4. DDP Performance Analysis

Key metric: **compute-to-communication ratio**. If all-reduce takes longer than the backward pass, you are communication-bound and adding GPUs doesn't help.

$$\text{Efficiency} = \frac{T_{\text{backward}}}{T_{\text{backward}} + T_{\text{all-reduce}} - T_{\text{overlap}}}$$

For a 7B model with 8 x A100 80GB:
- Backward pass: ~500ms (estimate)
- All-reduce volume: 2 * 7e9 * 2 bytes / 8 GPUs * (8-1)/8 ≈ 12 GB
- NVLink bandwidth: 600 GB/s -> all-reduce time: 12/600 = 20ms
- With full overlap: efficiency ≈ 500/(500+20-18) ≈ 96%

This is why DDP scales well up to ~8-16 GPUs on NVLink. Beyond that, you need InfiniBand (slower) and efficiency drops.


## Summary

### Key Concepts

| Concept | Detail | Performance impact |
|---------|--------|-------------------|
| Gradient bucketing | Accumulate to 25MB buckets, then all-reduce | Reduces N small -> few large comms |
| Overlap | All-reduce overlaps with backward | Near-zero overhead with good overlap |
| find_unused_parameters | Extra graph traversal | 10-30% overhead; only enable if needed |
| DistributedSampler | Partition data across ranks | Essential for correct training |
| bucket_cap_mb | Tune based on model and network | 25MB default is usually fine |

**Debugging DDP**:
- `NCCL_DEBUG=INFO`: verbose NCCL logging
- Training hangs: usually a NCCL timeout -- one rank crashed or is slow
- Loss diverges: likely missing `DistributedSampler` or seed issue

**Next**: `04_fsdp_and_zero_sharding.ipynb` -- training 100B+ parameter models by sharding everything.
